# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sagarjana00/FlyRank_AI_ML_Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My chosen lane is **Refresh / Content Opportunity Scoring**.

I frame this primarily as a **ranking/scoring problem**.

The goal is to assign an opportunity or review score to content pages and rank them so that a content or SEO team can review the most relevant pages first.

The output is therefore not simply a prediction of whether a page is "good" or "bad". It is intended to prioritize limited human review capacity.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## Target or proxy

For the starter dataset, I will use `trend_direction == "down"` as an initial **proxy target** for identifying pages showing downward movement.

This is only a proxy because `trend_direction` describes the current observation window rather than a future outcome.

A stronger future version of the problem could define a leakage-safe future target, such as using a prior feature window to predict whether a page declines during a later target window.

For example:

**Prior 90 days of features → next 30 days decline**

The future-looking target will require additional data preparation and leakage checks in later stages.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## Success metric

My primary success metric will be **Precision@K**, with **Precision@50** as an initial example of the review capacity.

Precision@50 measures how many of the top 50 pages selected by the ranking actually match the target or proxy.

This metric matches the real decision because the content team has limited review capacity and would act on the highest-ranked pages first.

I may also consider Average Precision and Recall in later validation, depending on the final target and review capacity.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [9]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
display(df.head())

Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [10]:
print("Rows:", len(df))
print("Unique content IDs:", df["content_id"].nunique())

Rows: 30000
Unique content IDs: 30000


In [11]:
display(
    df[
        [
            "content_id",
            "client_id",
            "impressions_90d",
            "sessions_90d",
            "content_age_days",
            "trend_direction"
        ]
    ].head(10)
)

,content_id,client_id,impressions_90d,sessions_90d,content_age_days,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,17,187,down
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,down
3,content_331d6c4de07b,client_19581e27de,11751,78,463,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,down
5,content_d4084a4bc775,client_f369cb89fc,3970,5,147,down
6,content_9a34b442b552,client_8722616204,20,1,90,down
7,content_a63219c6e95a,client_19581e27de,1724,28,445,stable
8,content_5e6c160719bc,client_6208ef0f77,32574,68,90,down
9,content_c27558df2b0c,client_19581e27de,1240,3,257,down


## Unit of analysis

The unit of analysis is **one content page/content item**.

The starter dataset contains 30,000 rows and 30,000 unique `content_id` values, so each row in this dataset represents one unique content item.

The model or scoring system would therefore produce a score or prediction for each content item, which could then be used to create a ranked review queue.


In [12]:
df["is_declining_proxy"] = (
    df["trend_direction"] == "down"
).astype(int)

display(
    df[
        [
            "content_id",
            "trend_direction",
            "is_declining_proxy"
        ]
    ].head(10)
)

,content_id,trend_direction,is_declining_proxy
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1
5,content_d4084a4bc775,down,1
6,content_9a34b442b552,down,1
7,content_a63219c6e95a,stable,0
8,content_5e6c160719bc,down,1
9,content_c27558df2b0c,down,1


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule is useful as a transparent baseline, but it may rely on only a small number of manually chosen conditions.

For example, a rule could prioritize every page with a downward trend. This would not necessarily distinguish between pages with different levels of impressions, engagement, position, freshness, or other observable signals.

ML could potentially combine multiple signals and learn patterns associated with the target or proxy. This may produce a more useful ranking of pages than a single manually defined rule.

However, ML should only be preferred if it demonstrates better performance on an appropriate validation setup. A more complex model is not automatically better than a simple rule.


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.